In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

In [8]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.head(5)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [9]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [10]:
df['industry'].mode()[0]

'technology'

In [11]:
# حساب مصفوفة الارتباط للأعمدة الرقمية المطلوبة
features = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']
corr_matrix = df[features].corr()

# عرض قيم الارتباط الخاصة بالخيارات الأربعة
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for f1, f2 in pairs:
    print(f"{f1} and {f2}: {corr_matrix.loc[f1, f2]:.4f}")

interaction_count and lead_score: 0.9287
number_of_courses_viewed and lead_score: 0.7694
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1064


In [20]:
# 1. التأكد من ملء الفراغات كما طلبت تعليمات الواجب
df['converted'] = df['converted'].fillna(0).astype(int)

features = ['industry', 'location', 'lead_source', 'employment_status']
for col in features:
    df[col] = df[col].fillna('NA').astype(str)

# 2. تقسيم البيانات (60% Train, 20% Val, 20% Test) مع random_state=42
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
y_train = df_train['converted'].values

# 3. حساب Mutual Information
def calculate_mi(series):
    return mutual_info_score(series, y_train)

#mi_scores = df_train[features].apply(calculate_mi)
#print(mi_scores.sort_values(ascending=False))
# حساب قيم الـ MI وترتيبها
mi_scores = df_train[features].apply(calculate_mi).sort_values(ascending=False)

print(mi_scores)
print("-" * 30)
print(f"Biggest MI Feature : {mi_scores.idxmax()}")
print(f"Max MI Value       : {mi_scores.max():.6f}")

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64
------------------------------
Biggest MI Feature : lead_source
Max MI Value       : 0.034346


In [22]:
# 1. تحديد الأعمدة وتجهيزها
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

# ملء الفراغات حسب شروط الواجب
categorical = list(df.select_dtypes(include=['object', 'str']).columns)
numerical = [c for c in df.select_dtypes(include=['number']).columns if c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
df['converted'] = df['converted'].fillna(0).astype(int)

# 2. تقسيم البيانات (60% Train, 20% Val, 20% Test) مع random_state=42
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# حذف عمود الهدف من مجموعات الميزات
del df_train['converted']
del df_val['converted']
del df_test['converted']

# 3. التحويل عبر DictVectorizer
features = categorical + numerical

train_dict = df_train[features].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

# 4. تدريب نموذج Logistic Regression
model = LogisticRegression(solver='liblinear', C=10, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 5. التنبؤ وحساب الدقة على بيانات التحقق (Validation)
y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {round(acc, 2)}")
print(f"Exact Accuracy: {acc:.4f}")

Validation Accuracy: 0.65
Exact Accuracy: 0.6450


In [23]:
# قائمة الميزات كاملة
features = categorical + numerical

# الدقة الأصلية
orig_score = acc

differences = {}

for f in features:
    # إنشاء قائمة الميزات بدون المتغير الحالي
    subset = [col for col in features if col != f]
    
    # تحويل بيانات التدريب والتحقق للمجموعة الفرعية
    train_dict = df_train[subset].to_dict(orient='records')
    X_tr = dv.fit_transform(train_dict)
    
    val_dict = df_val[subset].to_dict(orient='records')
    X_v = dv.transform(val_dict)
    
    # تدريب النموذج
    m = LogisticRegression(solver='liblinear', C=10, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    
    # التنبؤ وحساب الدقة الجديدة
    score = accuracy_score(y_val, m.predict(X_v))
    
    # حساب الفرق
    differences[f] = orig_score - score

# عرض الفروق مرتبة تصاعدياً (من الأقل تأثيراً إلى الأكثر تأثيراً)
import pandas as pd
diff_df = pd.DataFrame(list(differences.items()), columns=['feature', 'difference'])
diff_df['abs_difference'] = diff_df['difference'].abs()
diff_df = diff_df.sort_values(by='abs_difference')

print(diff_df.to_string(index=False))

                 feature  difference  abs_difference
                industry       0.000           0.000
       employment_status       0.000           0.000
                location       0.000           0.000
              lead_score       0.001           0.001
number_of_courses_viewed       0.002           0.002
             lead_source       0.003           0.003
       interaction_count       0.044           0.044
           annual_income      -0.082           0.082


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

c_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in c_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    score = accuracy_score(y_val, model.predict(X_val))
    print(f"C = {c:<9} -> Accuracy: {score:.4f} (rounded: {round(score, 3)})")

C = 1e-06     -> Accuracy: 0.5980 (rounded: 0.598)
C = 1e-05     -> Accuracy: 0.5980 (rounded: 0.598)
C = 0.0001    -> Accuracy: 0.6130 (rounded: 0.613)
C = 0.001     -> Accuracy: 0.6450 (rounded: 0.645)
